# 🧠 Proprietary Multilingual STT Model Training (From Scratch)
### 100% In-House PyTorch Architecture: Conv2d Subsampling + Transformer + CTC Loss

This notebook trains our **Proprietary Speech-to-Text Model** from scratch without relying on external pretrained models.

---
### ⚡ IMPORTANT KAGGLE SETUP:
1. **Enable GPU**: Right sidebar ➔ **Accelerator** ➔ Select **GPU T4 x 2** or **GPU P100**.
2. **Upload Dataset**: Upload `conversational_stt_dataset.zip` via **+ Add Data** or upload directly into `/kaggle/working`.
3. **Run All Cells**: Training takes ~35-45 minutes on GPU!

## Step 1: Check GPU & Install Required Libraries

In [ ]:
!nvidia-smi
!pip install -q soundfile librosa torchaudio
print('✅ Required libraries verified!')

## Step 2: Locate & Extract Dataset

In [ ]:
import os, zipfile, glob, shutil

WORKING_DIR = '/kaggle/working'
dataset_dir = os.path.join(WORKING_DIR, 'dataset')
os.makedirs(dataset_dir, exist_ok=True)

zip_candidates = glob.glob('/kaggle/input/**/conversational_stt_dataset*.zip', recursive=True) + \
                 glob.glob('/kaggle/working/**/conversational_stt_dataset*.zip', recursive=True) + \
                 glob.glob('/kaggle/input/**/colab_stt_dataset*.zip', recursive=True) + \
                 glob.glob('/kaggle/working/**/colab_stt_dataset*.zip', recursive=True) + \
                 glob.glob('/kaggle/input/**/*.zip', recursive=True)

target_zip = None
for z in zip_candidates:
    if 'stt' in z.lower() or 'dataset' in z.lower() or 'conversational' in z.lower():
        target_zip = z
        break

if target_zip and os.path.exists(target_zip):
    print(f'📦 Extracting: {target_zip}...')
    with zipfile.ZipFile(target_zip, 'r') as zf:
        zf.extractall(dataset_dir)
    print('✅ Extraction complete!')

manifest_candidates = glob.glob(f'{dataset_dir}/**/manifest*.csv', recursive=True) + \
                      glob.glob('/kaggle/input/**/manifest*.csv', recursive=True)
if not manifest_candidates:
    raise FileNotFoundError('❌ manifest.csv not found!')

manifest_path = manifest_candidates[0]
manifest_dir = os.path.dirname(manifest_path)
print(f'✅ Found manifest at: {manifest_path}')

## Step 3: Build Multilingual Character Vocabulary (Gujarati, Hindi, English)

In [ ]:
import re, json, pandas as pd

df = pd.read_csv(manifest_path)
print(f'Loaded {len(df):,} audio samples.')

class CTCTokenizer:
    PAD_TOKEN = '<pad>'
    UNK_TOKEN = '<unk>'
    BLANK_TOKEN = '<blank>'
    SPACE_TOKEN = '|'
    SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, BLANK_TOKEN, SPACE_TOKEN]

    def __init__(self, vocab=None):
        if vocab:
            self.char2id = dict(vocab)
        else:
            self.char2id = {token: idx for idx, token in enumerate(self.SPECIAL_TOKENS)}
        self.id2char = {idx: token for token, idx in self.char2id.items()}

    def clean_text(self, text):
        if not text:
            return ''
        text = str(text).lower().strip()
        text = re.sub(r'[^\w\s\u0A80-\u0AFF\u0900-\u097F]', '', text)
        text = re.sub(r'\s+', self.SPACE_TOKEN, text)
        return text

    def build_vocab(self, texts):
        unique_chars = set()
        for text in texts:
            cleaned = self.clean_text(text)
            for char in cleaned:
                if char != self.SPACE_TOKEN:
                    unique_chars.add(char)
        sorted_chars = sorted(list(unique_chars))
        self.char2id = {token: idx for idx, token in enumerate(self.SPECIAL_TOKENS)}
        for char in sorted_chars:
            if char not in self.char2id:
                self.char2id[char] = len(self.char2id)
        self.id2char = {idx: token for token, idx in self.char2id.items()}
        return self.char2id

    def encode(self, text):
        cleaned = self.clean_text(text)
        return [self.char2id.get(c, self.char2id[self.UNK_TOKEN]) for c in cleaned]

    def decode(self, token_ids, collapse_ctc=True):
        if collapse_ctc:
            collapsed = []
            prev_token = None
            blank_id = self.char2id.get(self.BLANK_TOKEN, 2)
            pad_id = self.char2id.get(self.PAD_TOKEN, 0)
            for tid in token_ids:
                if tid in (blank_id, pad_id):
                    prev_token = tid
                    continue
                if tid != prev_token:
                    collapsed.append(tid)
                    prev_token = tid
            chars = [self.id2char.get(tid, '') for tid in collapsed]
        else:
            chars = [self.id2char.get(tid, '') for tid in token_ids if tid not in (0, 1, 2)]
        return ''.join(chars).replace(self.SPACE_TOKEN, ' ').strip()

    def __len__(self):
        return len(self.char2id)

tokenizer = CTCTokenizer()
vocab = tokenizer.build_vocab(df['transcript'].dropna().tolist())
print(f'✅ Vocab built successfully! Total unique characters: {len(vocab)}')
with open('/kaggle/working/vocab.json', 'w', encoding='utf-8') as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

## Step 4: PyTorch STT Dataset Loader with On-The-Fly Log-Mel Extraction

In [ ]:
import torch, soundfile as sf, librosa, numpy as np
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader

class KaggleSTTDataset(Dataset):
    def __init__(self, df, manifest_dir, tokenizer, sample_rate=16000, n_mels=80):
        self.df = df.reset_index(drop=True)
        self.manifest_dir = manifest_dir
        self.tokenizer = tokenizer
        self.sample_rate = sample_rate
        self.mel_transform = T.MelSpectrogram(
            sample_rate=sample_rate, n_fft=400, win_length=400, hop_length=160, n_mels=n_mels
        )

    def resolve_path(self, p):
        p_str = str(p).strip()
        if os.path.exists(p_str): return p_str
        fname = os.path.basename(p_str)
        for root in [self.manifest_dir, os.path.join(self.manifest_dir, 'wav_chunks')]:
            c = os.path.join(root, fname)
            if os.path.exists(c): return c
        m = glob.glob(f'{self.manifest_dir}/**/{fname}', recursive=True)
        return m[0] if m else p_str

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = self.resolve_path(row['audio_path'])
        transcript = str(row['transcript']).strip()
        try:
            audio, sr = sf.read(audio_path)
            if audio.ndim > 1: audio = audio.mean(axis=1)
            if sr != self.sample_rate:
                audio = librosa.resample(audio, orig_sr=sr, target_sr=self.sample_rate)
        except Exception:
            audio = np.zeros(self.sample_rate, dtype=np.float32)

        waveform = torch.from_numpy(audio).float()
        if waveform.abs().max() > 0:
            waveform = (waveform / waveform.abs().max()) * 0.95
        mel = self.mel_transform(waveform)
        log_mel = torch.log(torch.clamp(mel, min=1e-5))
        mean, std = log_mel.mean(dim=-1, keepdim=True), log_mel.std(dim=-1, keepdim=True) + 1e-5
        norm_mel = (log_mel - mean) / std
        token_ids = self.tokenizer.encode(transcript)
        return {'mel': norm_mel, 'target': torch.tensor(token_ids, dtype=torch.long), 'transcript': transcript}

def collate_fn_ctc(batch):
    mel_lens = [b['mel'].shape[1] for b in batch]
    target_lens = [len(b['target']) for b in batch]
    max_mel_len = max(mel_lens)
    max_tgt_len = max(target_lens) if max(target_lens) > 0 else 1

    padded_mels = torch.zeros(len(batch), 80, max_mel_len)
    padded_targets = torch.zeros(len(batch), max_tgt_len, dtype=torch.long)
    for i, b in enumerate(batch):
        padded_mels[i, :, :mel_lens[i]] = b['mel']
        if len(b['target']) > 0:
            padded_targets[i, :len(b['target'])] = b['target']

    return {
        'features': padded_mels,
        'feature_lengths': torch.tensor(mel_lens, dtype=torch.long),
        'targets': padded_targets,
        'target_lengths': torch.tensor(target_lens, dtype=torch.long),
        'transcripts': [b['transcript'] for b in batch]
    }

# 90/10 Split
train_df = df.sample(frac=0.90, random_state=42)
val_df = df.drop(train_df.index)

train_loader = DataLoader(KaggleSTTDataset(train_df, manifest_dir, tokenizer), batch_size=16, shuffle=True, collate_fn=collate_fn_ctc, num_workers=2)
val_loader = DataLoader(KaggleSTTDataset(val_df, manifest_dir, tokenizer), batch_size=16, shuffle=False, collate_fn=collate_fn_ctc, num_workers=2)
print(f'Train batches: {len(train_loader)} | Validation batches: {len(val_loader)}')

## Step 5: Define Proprietary STT Model Architecture (Pure PyTorch)

In [ ]:
import math
import torch.nn as nn
import torch.nn.functional as F

class Conv2dSubsampling(nn.Module):
    def __init__(self, input_dim=80, hidden_dim=256, conv_channels=256):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, conv_channels, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(conv_channels, conv_channels, kernel_size=3, stride=2, padding=1),
            nn.ReLU()
        )
        self.out_proj = nn.Linear(conv_channels * (input_dim // 4), hidden_dim)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv(x)
        b, c, f, t = x.size()
        x = x.permute(0, 3, 1, 2).contiguous().view(b, t, c * f)
        return self.out_proj(x)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class ProprietarySTTModel(nn.Module):
    def __init__(self, input_dim=80, hidden_dim=256, num_layers=6, num_heads=4, vocab_size=200, dropout=0.1, blank_id=2):
        super().__init__()
        self.blank_id = blank_id
        self.subsampling = Conv2dSubsampling(input_dim, hidden_dim)
        self.pos_encoder = PositionalEncoding(hidden_dim, dropout=dropout)
        enc_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=num_heads, dim_feedforward=hidden_dim*4, dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.ctc_linear = nn.Linear(hidden_dim, vocab_size)

    def forward(self, features, feature_lengths):
        x = self.subsampling(features)
        out_lengths = (feature_lengths // 4).clamp(min=1)
        x = self.pos_encoder(x)
        max_len = x.size(1)
        mask = torch.arange(max_len, device=x.device).unsqueeze(0) >= out_lengths.unsqueeze(1)
        encoded = self.layer_norm(self.transformer(x, src_key_padding_mask=mask))
        logits = self.ctc_linear(encoded)
        return F.log_softmax(logits, dim=-1), out_lengths

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ProprietarySTTModel(vocab_size=len(tokenizer)).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Proprietary STT Model loaded on {device}! Total Trainable Parameters: {total_params:,}')

## Step 6: Train Proprietary STT Model from Scratch (60 Epochs)

In [ ]:
import time

NUM_EPOCHS = 60
criterion = nn.CTCLoss(blank=model.blank_id, zero_infinity=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=1e-4, betas=(0.9, 0.98))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

best_val_loss = float('inf')
SAVE_PATH = '/kaggle/working/best_proprietary_model.pt'

print(f'🚀 Starting Training for {NUM_EPOCHS} Epochs on {device}...')
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    t0 = time.time()

    for step, batch in enumerate(train_loader):
        features = batch['features'].to(device)
        f_lens = batch['feature_lengths'].to(device)
        targets = batch['targets'].to(device)
        t_lens = batch['target_lengths'].to(device)

        optimizer.zero_grad()
        log_probs, enc_lens = model(features, f_lens)
        loss = criterion(log_probs.permute(1, 0, 2), targets, enc_lens, t_lens)

        if not (torch.isnan(loss) or torch.isinf(loss)):
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item()

    scheduler.step()
    train_loss = total_loss / max(1, len(train_loader))

    # Validation
    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            features, f_lens = batch['features'].to(device), batch['feature_lengths'].to(device)
            targets, t_lens = batch['targets'].to(device), batch['target_lengths'].to(device)
            log_probs, enc_lens = model(features, f_lens)
            loss = criterion(log_probs.permute(1, 0, 2), targets, enc_lens, t_lens)
            if not (torch.isnan(loss) or torch.isinf(loss)):
                v_loss += loss.item()
    val_loss = v_loss / max(1, len(val_loader))

    elapsed = round(time.time() - t0, 1)
    print(f'Epoch [{epoch:02d}/{NUM_EPOCHS}] in {elapsed}s | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {optimizer.param_groups[0]["lr"]:.2e}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'vocab_size': len(tokenizer),
            'val_loss': val_loss
        }, SAVE_PATH)

print(f'🎉 Training Complete! Best Validation Loss: {best_val_loss:.4f}')

## Step 7: Test Inference on Live Validation Clips & Package 1-Click ZIP

In [ ]:
# Load best checkpoint
checkpoint = torch.load(SAVE_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print('=== SAMPLE INFERENCE ON VALIDATION AUDIO ===')
val_batch = next(iter(val_loader))
with torch.no_grad():
    log_probs, enc_lens = model(val_batch['features'].to(device), val_batch['feature_lengths'].to(device))
    preds = torch.argmax(log_probs, dim=-1).cpu().numpy()

for i in range(min(5, len(val_batch['transcripts']))):
    tokens = preds[i][:enc_lens[i].item()]
    hyp = tokenizer.decode(tokens, collapse_ctc=True)
    ref = val_batch['transcripts'][i]
    print(f'\n[Sample {i+1}]')
    print('  Expected:   ', ref)
    print('  Proprietary:', hyp)

# Package model & vocab into zip for 1-click download
EXPORT_ZIP = '/kaggle/working/proprietary_stt_model.zip'
with zipfile.ZipFile(EXPORT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(SAVE_PATH, arcname='proprietary_stt_model.pt')
    zf.write('/kaggle/working/vocab.json', arcname='vocab.json')

size_mb = os.path.getsize(EXPORT_ZIP) / (1024 * 1024)
print(f'\n🎉 Model packaged: {EXPORT_ZIP} ({size_mb:.2f} MB)')
print('👉 In Kaggle right sidebar: Output section ➔ Click 3 dots on proprietary_stt_model.zip ➔ Click Download!')